# 01 · Contrato de datos, partición por paciente y caché preprocesado

Flujo (SI7011): contrato → duplicados → partición **por paciente** → acondicionamiento → estadísticas de etiquetas **solo con train**.
Entradas permitidas: PPG y ECG. ABP produce las etiquetas, así que nunca se usa como entrada.

In [1]:
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "bp").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, torch
print("ROOT:", ROOT, "| cuda:", torch.cuda.is_available())

ROOT: C:\Users\USUARIO\Documents\Source\dl_bp_mimic_proyecto_integrador | cuda: True


In [2]:
from bp.contract import (read_subject_ids, check_patient, duplicate_report, build_data_card, save_data_card,
                         MIN_VALID_SEGMENTS)
from bp.paths import DATA, DATA_CARD, SPLIT_JSON, PREPROCESSED, ARTIFACTS

ids = read_subject_ids()
print("IDs en subject_ids.txt:", len(ids))
present = {m: {p.name.split('_')[0] for p in (DATA / m).glob('*.npy')} for m in ('ppg', 'ecg', 'labels')}
for m, s in present.items():
    assert set(ids) == s, f"{m}: los archivos no coinciden con subject_ids.txt"
print("ppg / ecg / labels cubren exactamente los mismos pacientes")

IDs en subject_ids.txt: 1524
ppg / ecg / labels cubren exactamente los mismos pacientes


## 1. Contrato por paciente
Política: un segmento inválido se descarta; el paciente se excluye si es ilegible o le quedan menos de 15 segmentos válidos.

In [3]:
t0 = time.time()
reports = [check_patient(s) for s in ids]
masks = {r['subject_id']: r['valid_mask'] for r in reports}
excluded = [{"subject_id": r['subject_id'], "reason": "; ".join(r['issues']) or f"{r['n_valid']} segmentos válidos"}
            for r in reports if r['issues'] or r['n_valid'] < MIN_VALID_SEGMENTS]
kept = [s for s in ids if s not in {e['subject_id'] for e in excluded}]
n_valid = sum(int(masks[s].sum()) for s in kept)
print(f"{time.time()-t0:.0f}s | pacientes conservados {len(kept)}/{len(ids)} | excluidos {len(excluded)} | "
      f"segmentos válidos {n_valid} | segmentos descartados {sum(int((~masks[s]).sum()) for s in kept)}")

4s | pacientes conservados 1524/1524 | excluidos 0 | segmentos válidos 45720 | segmentos descartados 0


## 2. Duplicados (antes de partir)
SHA-1 sobre cada segmento (PPG+ECG) para detectar réplicas entre pacientes.

In [4]:
dup = duplicate_report(kept, mask_by_id=masks)
print(dup)
# Política: se descarta la réplica (el primer paciente conserva el segmento) y se recalcula todo antes de partir.
for sid, segs in dup['drop_segments'].items():
    masks[sid] = masks[sid].copy(); masks[sid][segs] = False
dup_after = duplicate_report(kept, mask_by_id=masks)
assert dup_after['n_segments_dropped'] == 0, dup_after
kept = [s for s in kept if masks[s].sum() >= MIN_VALID_SEGMENTS]
print("pacientes:", len(kept), "| segmentos válidos tras quitar duplicados:", sum(int(masks[s].sum()) for s in kept))

{'drop_segments': {'p042858': [14, 17, 25, 26, 28], 'p062561': [4, 22, 23, 24, 27, 29], 'p083120': [3], 'p085508': [13, 15, 17, 18, 20, 22, 25, 26, 29], 'p090269': [17, 18, 20, 21, 22, 24, 26, 27, 29]}, 'n_segments_dropped': 30, 'cross_patient_duplicates': {'p083120': ['p072885']}, 'n_patients_with_cross_duplicates': 1}


pacientes: 1524 | segmentos válidos tras quitar duplicados: 45690


## 3. Partición 60/20/20 por paciente (seed=0)

In [5]:
from bp.split import make_split, save_split, assert_no_leak
split = make_split(kept, seed=0)
assert_no_leak(split)
assert len(set(split['train']) & set(split['test'])) == 0
save_split(split)
for k, v in split.items():
    print(k, len(v), "pacientes", sum(int(masks[s].sum()) for s in v), "segmentos")

train 914 pacientes 27396 segmentos
val 305 pacientes 9149 segmentos
test 305 pacientes 9145 segmentos


## 4. Acondicionamiento: Butterworth pasa-banda de fase cero + z-score por segmento
Se preprocesa una sola vez, en float64, y se guarda en float32.

In [6]:
from bp.preprocess import build_cache, preprocess_signal
t0 = time.time()
info = build_cache(kept, masks)
print(info, f"{time.time()-t0:.0f}s")
for f in sorted(PREPROCESSED.glob('*.npy')):
    print(f.name, f"{f.stat().st_size/1e6:.0f} MB")

{'n_segments': 45690, 'n_subjects': 1524} 48s
ecg.npy 685 MB
labels.npy 0 MB
ppg.npy 685 MB


In [7]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sid = split['train'][0]
raw_p = np.load(DATA/'ppg'/f'{sid}_ppg.npy')[0]; raw_e = np.load(DATA/'ecg'/f'{sid}_ecg.npy')[0]
fp = preprocess_signal(raw_p, 'ppg')[0]; fe = preprocess_signal(raw_e, 'ecg')[0]
t = np.arange(3750)/125
fig, ax = plt.subplots(2, 2, figsize=(12, 5), sharex='col')
for j, (raw, f, n) in enumerate([(raw_p, fp, 'PPG'), (raw_e, fe, 'ECG')]):
    ax[0, j].plot(t, raw, lw=.6); ax[0, j].set_title(f'{n} crudo'); ax[1, j].plot(t, f, lw=.6); ax[1, j].set_title(f'{n} filtrado + z-score')
    ax[1, j].set_xlabel('s')
plt.tight_layout(); plt.savefig(ARTIFACTS/'fig_preprocesamiento.png', dpi=110)
print("media/std del segmento filtrado:", fp.mean().round(4), fp.std().round(4))

media/std del segmento filtrado: 0.0 1.0


## 5. Etiquetas: estadísticas de normalización **solo con train**

In [8]:
from bp.split import load_split
from bp.dataset import split_rows, label_stats
from bp.experiment import save_label_stats
labels = np.load(PREPROCESSED/'labels.npy')
tr_rows, _ = split_rows(load_split()['train'])
y_mean, y_std = label_stats(labels[tr_rows])
save_label_stats(y_mean, y_std)
print("train  PAS/PAD media", y_mean.round(2), "DE", y_std.round(2))
for k in ('val', 'test'):
    r, _ = split_rows(load_split()[k]); print(k, "media", labels[r].mean(0).round(2), "DE", labels[r].std(0).round(2))

train  PAS/PAD media [112.78  58.48] DE [18.14 12.51]
val media [112.83  58.78] DE [17.46 12.07]
test media [112.92  58.01] DE [17.28 11.66]


## 6. data_card.json

In [9]:
card = build_data_card(ids, kept, masks, excluded, dup, labels)
card['split'] = {k: {'n_subjects': len(v), 'n_segments': sum(int(masks[s].sum()) for s in v)} for k, v in split.items()}
card['split_seed'] = 0
card['preprocessing'] = {'filters': {'order': 4, 'ppg_band_hz': [0.5, 8.0], 'ecg_band_hz': [0.5, 40.0], 'zero_phase': True, 'fs': 125},
                         'input_norm': 'z-score por segmento (por instancia)', 'label_norm': 'z-score con media/DE de train'}
save_data_card(card)
print(json.dumps({k: card[k] for k in ('n_subjects_kept','n_segments_valid','split','label_descriptives_mmHg')}, indent=1))

{
 "n_subjects_kept": 1524,
 "n_segments_valid": 45690,
 "split": {
  "train": {
   "n_subjects": 914,
   "n_segments": 27396
  },
  "val": {
   "n_subjects": 305,
   "n_segments": 9149
  },
  "test": {
   "n_subjects": 305,
   "n_segments": 9145
  }
 },
 "label_descriptives_mmHg": {
  "pas": {
   "mean": 112.81552124023438,
   "std": 17.83379364013672,
   "min": 63.79999923706055,
   "max": 185.8000030517578,
   "p5": 85.9000015258789,
   "p95": 143.8000030517578
  },
  "pad": {
   "mean": 58.44516372680664,
   "std": 12.258400917053223,
   "min": 31.100000381469727,
   "max": 115.5999984741211,
   "p5": 40.79999923706055,
   "p95": 80.69999694824219
  }
 }
}
